In [1]:
import math

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self._prev = set(_children)
        self._backward = lambda: None
        self._op = _op
        self.label = label
        self.grad = 0.0

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            ''' backward function is always grad of out wrt self * out.grad
            helps to write equation out = self + other. Then find partial L / partial self
            '''
            self.grad = 1.0 * out.grad
            other.grad = 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad = other.data * out.grad
            other.grad = self.data * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        n = self.data
        t = (math.exp(2*n) - 1) / (math.exp(2*n) + 1)
        out = Value(t, (self,), 'tanh')

        def _backward():
            self.grad = (1-t**2) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(o)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()
        

In [3]:
a = Value(2.0, label = 'a')
b = Value(-3.0, label = 'b')
c = Value(10.0, label = 'c')

In [4]:
e = a*b; e.label = 'e'
d = e + c; d.label = 'd'
f = Value(-2.0, label='f')
L = d * f; L.label = 'L'

In [5]:
import re
from graphviz import Digraph


class ScrollableGraph:
    """
    wraps a Digraph so the notebook shows it at full size in a scroll box
    instead of shrinking the whole graph down to the cell width

    ctrl/cmd + wheel over the graph zooms (where the notebook runs output js)
    """
    def __init__(self, dot, height=520, scale=1.0):
        self.dot = dot
        self.height = height   # px, height of the scroll box
        self.scale = scale     # 1.0 = graphviz's natural size

    def __getattr__(self, name):
        # hide graphviz's own repr hooks: its _repr_mimebundle_ would otherwise be
        # found here and win over _repr_html_, so the wrapper would do nothing
        if name.startswith('_repr_') or name.startswith('_ipython_'):
            raise AttributeError(name)
        # anything else (render, source, view, ...) falls through to the Digraph
        return getattr(self.dot, name)

    def _repr_html_(self):
        svg = self.dot.pipe(format='svg').decode('utf-8')
        svg = svg[svg.index('<svg'):]  # drop the xml/doctype preamble so the svg can be inlined

        # graphviz sizes the svg in points; browsers render 1pt as 4/3 px
        w, h = (float(v) for v in re.search(
            r'width="([\d.]+)pt"\s+height="([\d.]+)pt"', svg).groups())
        w, h = round(w * 4 / 3 * self.scale), round(h * 4 / 3 * self.scale)

        # let the svg fill the sized inner div rather than carry its own dimensions:
        # notebook css often forces `svg {max-width: 100% !important}`, which would
        # otherwise shrink the graph back to the cell width and leave nothing to scroll
        svg = re.sub(r'width="[\d.]+pt"\s+height="[\d.]+pt"',
                     'width="100%" height="100%" preserveAspectRatio="xMinYMin meet"',
                     svg, count=1)

        uid = 'g%d' % id(self)
        # the width must be a *definite* length: vs code renders output in an
        # auto-sized iframe, so `width:100%` resolves against a shrink-to-fit box
        # and the div just grows to the graph's width -> clipped, no scrollbar.
        # 1vw is the output viewport, which is a real width, so overflow can kick in
        return f'''
<div id="{uid}" style="overflow:auto !important; width:calc(100vw - 48px); max-width:{w}px;
     max-height:{self.height}px; resize:both; border:1px solid #ddd; border-radius:4px;
     background:#fff; box-sizing:border-box;">
  <div style="width:{w}px; height:{h}px;" id="{uid}-inner">{svg}</div>
</div>
<script>
(function() {{
  var box = document.getElementById("{uid}"), inner = document.getElementById("{uid}-inner");
  if (!box || !inner) return;
  var w = {w}, h = {h}, z = 1;
  box.addEventListener("wheel", function(e) {{
    if (!e.ctrlKey && !e.metaKey) return;   // plain wheel keeps scrolling the box
    e.preventDefault();
    z = Math.min(4, Math.max(0.2, z * (e.deltaY < 0 ? 1.1 : 1 / 1.1)));
    inner.style.width = (w * z) + "px";
    inner.style.height = (h * z) + "px";
    box.style.maxWidth = (w * z) + "px";
  }}, {{passive: false}});
}})();
</script>'''


def trace(root):
    """
    builds a set of all nodes and edges in a graph
    """
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root, height=520, scale=1.0):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
    nodes, edges = trace(root)    
    for n in nodes:
        uid = str(id(n))
        # for any value in the graph,create a rectangular ('record') node for it
        dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f}" % (n.label, n.data, n.grad), shape='record')
        if n._op:
            # if this value is a result of some operation, create an op node for it
            dot.node(name = uid + n._op, label=n._op)
            # and connect this node to it
            dot.edge(uid + n._op, uid)
    
    for n1, n2 in edges:
        # connect n1 to the op node of n2
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    
    return ScrollableGraph(dot, height=height, scale=scale)


In [6]:
draw_dot(L)

In [7]:
# inputs x1,x2
x1 = Value(2.0,label='x1')
x2 = Value(0.0,label='x2')
# weights w1, w2
w1 = Value(-3.0,label='w1')
w2 = Value(1.0,label='w2')
# bias of the neuron
b = Value(6.8813735870195432,label='b')
# x1*w1 + x2*w2 + b
x1w1 = x1*w1; x1w1.label ='x1w1'
x2w2 = x2*w2; x2w2.label ='x2w2'
x1w1x2w2 = x1w1+x2w2; x1w1x2w2.label='x1w1 + x2w2'
n = x1w1x2w2+b; n.label='n'
o = n.tanh() 
o.label = 'o'

In [15]:
draw_dot(o)

In [13]:
o.grad = 1
topo = []
visited = set()
def build_topo(v):
  if v not in visited:
    visited.add(v)
    for child in v._prev:
      build_topo(child)
    topo.append(v)
build_topo(o)

for node in reversed(topo):
  node._backward()

In [14]:
x2._backward()
x2.grad

0.4999999999999999